# Fase 3b + Fase 4 — Pipeline + Modelagem (Padrão sênior SEM LEAKAGE)

Regra de OURO deste notebook:
1. **TREINO (X_train / y_train, 112.500)** → split interno, CV 5-fold, tuning de hparams, seleção de modelo, threshold ótimo 10:1.
2. **HOLDOUT (X_test / y_test, 37.500)** → **lido para métricas SOMENTE NA ÚLTIMA CÉLULA**, UMA ÚNICA VEZ.

Seed = 42. Custo FN = 10 × FP = 10:1.

Todas as decisões podem ser reproduzidas rodando `src/models/fase04_modelagem_sem_leakage.py`.

## 0. Imports e paths

In [ ]:
import os, sys, pickle, warnings
import numpy as np, pandas as pd
from copy import deepcopy

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score,
    recall_score, accuracy_score, brier_score_loss, confusion_matrix
)
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

SRC = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), 'src'))
sys.path.insert(0, SRC)
from features.preprocessamento import carregar_params, SEED_DEFAULT
from features.pipeline_modelo import montar_pipeline, salvar_pipeline, obter_nomes_features

warnings.filterwarnings('ignore')
np.random.seed(SEED_DEFAULT)

BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC    = os.path.join(BASE, 'data', 'processed')
MODELOS = os.path.join(BASE, 'models')
PARAMS  = carregar_params(os.path.join(MODELOS, 'preprocessamento_params.pkl'))
print('SEED =', SEED_DEFAULT, '| scale_pos_weight esperado ≈ 13.96')

## 1. Carregamento das bases — TESTE fica "fechado a sete chaves" até a célula final

In [ ]:
X_train = pd.read_csv(os.path.join(PROC, 'X_train.csv'))
y_train = pd.read_csv(os.path.join(PROC, 'y_train.csv'))['inadimplente_2anos'].astype(int).values
X_test  = pd.read_csv(os.path.join(PROC, 'X_test.csv'))
y_test  = pd.read_csv(os.path.join(PROC, 'y_test.csv'))['inadimplente_2anos'].astype(int).values

NEG, POS = int((y_train == 0).sum()), int((y_train == 1).sum())
SPW = round(NEG / POS, 2)
CUSTO_FN, CUSTO_FP = 10, 1
f'TREINO={X_train.shape[0]:,}  NEG={NEG:,}  POS={POS:,}  SPW={SPW}  TESTE (fechado)={X_test.shape[0]:,}'

## 2. Helpers para CV 5-fold out-of-fold SOMENTE no TREINO

In [ ]:
def calcular_custo(y_true, y_pred, fn=CUSTO_FN, fp=CUSTO_FP):
    tn, fp_c, fn_c, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return int(fn*fn_c + fp*fp_c), int(fn_c), int(fp_c)

def cv_oof_scores(nome, modelo_base, grid_linha: dict):
    """StratifiedKFold 5 SÓ no TREINO. Não lê y_test em momento algum."""
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED_DEFAULT)
    folds_roc, folds_f1, folds_acc, folds_custo = [], [], [], []
    oof_proba = np.zeros(len(y_train), dtype=float)
    for tr_idx, va_idx in skf.split(X_train, y_train):
        pipe = montar_pipeline(deepcopy(modelo_base), PARAMS)
        pipe.fit(X_train.iloc[tr_idx], y_train[tr_idx])
        proba = pipe.predict_proba(X_train.iloc[va_idx])[:, 1]
        pred  = pipe.predict(X_train.iloc[va_idx])
        folds_roc.append(roc_auc_score(y_train[va_idx], proba))
        folds_f1.append(f1_score(y_train[va_idx], pred, zero_division=0))
        folds_acc.append(accuracy_score(y_train[va_idx], pred))
        folds_custo.append(calcular_custo(y_train[va_idx], pred)[0])
        oof_proba[va_idx] = proba
    return dict(
        Modelo=nome, **grid_linha,
        cv_roc_medio=float(np.mean(folds_roc)),
        cv_roc_std=float(np.std(folds_roc)),
        cv_f1_medio=float(np.mean(folds_f1)),
        cv_custo_somado=int(np.sum(folds_custo)),
        oof_proba=oof_proba
    )

def melhor_th_oof(oof_proba, y_true, ths=np.linspace(0.01, 0.99, 9801)):
    mc, mt = 1e18, None
    for th in ths:
        c = calcular_custo(y_true, (oof_proba>=th).astype(int))[0]
        if c < mc: mc, mt = c, th
    return round(float(mt), 3), int(mc)

'helpers OK'

## 3. Tuning SÓ com CV no TREINO

3.1 DecisionTree (depth 1..15, min_samples_leaf=50)

3.2 RandomForest (grid 6 depths × 3 class_weights, n_estimators=300, min_samples_leaf=30)

3.3 XGBoost (grid 3md × 3lr × 2mcw = 18)

3.4 LightGBM (grid 3md × 3lr × 2mcs = 18)

In [ ]:
# 3.1 DecisionTree
rows_dt = []
for d in range(1, 16):
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED_DEFAULT,
                               class_weight=None, min_samples_leaf=50)
    rows_dt.append(cv_oof_scores(f'DT d={d}', m, {'max_depth': d}))
best_dt_row = rows_dt[int(np.argmax([r['cv_roc_medio'] for r in rows_dt]))]
BEST_DT_DEPTH = int(best_dt_row['max_depth'])
print(f'🏆 DT vencedora: depth={BEST_DT_DEPTH}  CV ROC médio={best_dt_row["cv_roc_medio"]:.5f}')

# 3.2 RandomForest
cw_grid = [('balanced','balanced'), ('balanced_subs','balanced_subsample'),
           ('balanced_1_14', {0:1, 1:14})]
rows_rf = []
for d in [6, 8, 10, 12, 15, None]:
    for cw_l, cw_v in cw_grid:
        m = RandomForestClassifier(n_estimators=300, max_depth=d, class_weight=cw_v,
                                   n_jobs=-1, random_state=SEED_DEFAULT,
                                   min_samples_leaf=30, oob_score=False)
        rows_rf.append(cv_oof_scores(f'RF d={d} cw={cw_l}', m,
                                     {'max_depth':str(d), 'class_weight':cw_l}))
best_rf_row = rows_rf[int(np.argmax([r['cv_roc_medio'] for r in rows_rf]))]
best_rf_depth_s = best_rf_row['max_depth']
best_rf_depth   = None if best_rf_depth_s == 'None' else int(best_rf_depth_s)
best_rf_cw_l    = best_rf_row['class_weight']
BEST_RF_CW = [v for (l,v) in cw_grid if l == best_rf_cw_l][0]
print(f'🏆 RF vencedor: depth={best_rf_depth_s} cw={best_rf_cw_l}  CV ROC={best_rf_row["cv_roc_medio"]:.5f}')

# 3.3 XGBoost
rows_xgb = []
for md in (4, 5, 6):
    for lr in (0.03, 0.05, 0.07):
        for mcw in (80, 100):
            m = xgb.XGBClassifier(n_estimators=500, max_depth=md, learning_rate=lr,
                                  min_child_weight=mcw, subsample=0.9,
                                  colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1.0,
                                  scale_pos_weight=SPW, random_state=SEED_DEFAULT,
                                  n_jobs=-1, eval_metric='auc', tree_method='hist')
            rows_xgb.append(cv_oof_scores(f'XGB md={md} lr={lr} mcw={mcw}', m,
                                          {'max_depth':md,'learning_rate':lr,'min_child_weight':mcw}))
best_xgb_row = rows_xgb[int(np.argmax([r['cv_roc_medio'] for r in rows_xgb]))]
BEST_XGB = dict(max_depth=int(best_xgb_row['max_depth']),
                learning_rate=float(best_xgb_row['learning_rate']),
                min_child_weight=int(best_xgb_row['min_child_weight']))
print(f'🏆 XGB vencedor: {BEST_XGB}  CV ROC={best_xgb_row["cv_roc_medio"]:.5f}')

# 3.4 LightGBM
rows_lgb = []
for md in (5, 6, 7):
    for lr in (0.03, 0.05, 0.07):
        for mcs in (100, 150):
            m = lgb.LGBMClassifier(n_estimators=500, learning_rate=lr, max_depth=md,
                                   num_leaves=2**md-1, min_child_samples=mcs, subsample=0.9,
                                   colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1.0,
                                   scale_pos_weight=SPW, random_state=SEED_DEFAULT,
                                   n_jobs=-1, verbose=-1)
            rows_lgb.append(cv_oof_scores(f'LGB md={md} lr={lr} mcs={mcs}', m,
                                          {'max_depth':md,'learning_rate':lr,'min_child_samples':mcs}))
best_lgb_row = rows_lgb[int(np.argmax([r['cv_roc_medio'] for r in rows_lgb]))]
BEST_LGB = dict(max_depth=int(best_lgb_row['max_depth']),
                learning_rate=float(best_lgb_row['learning_rate']),
                min_child_samples=int(best_lgb_row['min_child_samples']),
                num_leaves=2**int(best_lgb_row['max_depth'])-1)
print(f'🏆 LGB vencedor: {BEST_LGB}  CV ROC={best_lgb_row["cv_roc_medio"]:.5f}')

## 4. Threshold ótimo para custo 10:1 — SÓ no OOF do TREINO

In [ ]:
ths = {}
for nome, row in [('DecisionTree best', best_dt_row),
                  ('RandomForest best', best_rf_row),
                  ('XGBoost best',      best_xgb_row),
                  ('LightGBM best',     best_lgb_row)]:
    th, c = melhor_th_oof(row['oof_proba'], y_train)
    ths[nome] = th
    print(f'{nome}: threshold ótimo (treino OOF)={th}  |  custo somado={c:,}')
with open(os.path.join(MODELOS, 'thresholds_otimos_custo_10_1.pkl'), 'wb') as f:
    pickle.dump(ths, f)
'thresholds pickle salvo. O HOLDOUT ainda não foi lido para nada.'

## 5. Refit final em 100% do TREINO com os hparams vencedores do CV

In [ ]:
def finalizadores():
    return {
        'Dummy (prior)': DummyClassifier(strategy='prior', random_state=SEED_DEFAULT),
        f'DecisionTree best (d={BEST_DT_DEPTH})':
            DecisionTreeClassifier(max_depth=BEST_DT_DEPTH, random_state=SEED_DEFAULT,
                                   class_weight=None, min_samples_leaf=50),
        f'RandomForest best (d={best_rf_depth_s}, cw={best_rf_cw_l})':
            RandomForestClassifier(n_estimators=300, max_depth=best_rf_depth,
                                   class_weight=BEST_RF_CW, n_jobs=-1,
                                   random_state=SEED_DEFAULT, min_samples_leaf=30),
        f'XGBoost best ({BEST_XGB})':
            xgb.XGBClassifier(n_estimators=500, **BEST_XGB, subsample=0.9,
                              colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1.0,
                              scale_pos_weight=SPW, random_state=SEED_DEFAULT, n_jobs=-1,
                              eval_metric='auc', tree_method='hist'),
        f'LightGBM best ({BEST_LGB})':
            lgb.LGBMClassifier(n_estimators=500, **BEST_LGB, subsample=0.9,
                               colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1.0,
                               scale_pos_weight=SPW, random_state=SEED_DEFAULT,
                               n_jobs=-1, verbose=-1),
    }

pipes = {}
for n, mc in finalizadores().items():
    p = montar_pipeline(mc, PARAMS); p.fit(X_train, y_train); pipes[n] = p
    print('✓ fit final TREINO:', n)

for arq, n in [('01_dummy_pipeline.pkl', f'Dummy (prior)'),
               ('02_decisiontree_best_pipeline.pkl', f'DecisionTree best (d={BEST_DT_DEPTH})'),
               ('03_randomforest_best_pipeline.pkl',
                f'RandomForest best (d={best_rf_depth_s}, cw={best_rf_cw_l})'),
               ('04_xgboost_pipeline.pkl', f'XGBoost best ({BEST_XGB})'),
               ('05_lightgbm_pipeline.pkl', f'LightGBM best ({BEST_LGB})')]:
    salvar_pipeline(pipes[n], os.path.join(MODELOS, arq))
'pickles salvos — HOLDOUT ainda fechado.'

## 6. 🥇🥈🥉  BATISMO do HOLDOUT — ÚNICA VEZ que o TESTE é lido

In [ ]:
linhas = []
c_pol, fn_pol, fp_pol = calcular_custo(y_test, np.zeros(len(y_test), dtype=int))
print(f'Política ATUAL (aprova todos) → CUSTO={c_pol:,} FN={fn_pol:,} FP={fp_pol:,}')
for n, p in pipes.items():
    proba = p.predict_proba(X_test)[:,1]
    pred_05 = (proba >= 0.5).astype(int)
    nome_curto = ([k for k in ths if n.startswith(k.split()[0])] + [None])[0]
    th_ot = ths.get(nome_curto, 0.5)
    pred_ot = (proba >= th_ot).astype(int)
    c_05, fn_05, fp_05 = calcular_custo(y_test, pred_05)
    c_ot, fn_ot, fp_ot = calcular_custo(y_test, pred_ot)
    linhas.append(dict(
        Modelo=n, Th_ótimo_treino_OOF=th_ot,
        ROC_AUC_teste=round(roc_auc_score(y_test, proba), 5),
        PR_AUC_teste=round(average_precision_score(y_test, proba), 5),
        F1_teste_th05=round(f1_score(y_test, pred_05, zero_division=0), 5),
        Custo_10_1_th05=c_05,
        Economia_th05_vs_política=round((1-c_05/c_pol)*100,2),
        Custo_10_1_th_ÓTIMO=c_ot,
        Economia_th_ÓTIMO_vs_política=round((1-c_ot/c_pol)*100,2),
    ))

tab = pd.DataFrame(linhas).sort_values('ROC_AUC_teste', ascending=False).reset_index(drop=True)
print(tab.to_string(index=False))
v = tab.iloc[0]
print(f"\n🏆 VENCEDOR do batismo: {v['Modelo']}")
print(f"   ROC AUC = {v['ROC_AUC_teste']:.5f} | Meta ≥ 0,85: {'SIM' if v['ROC_AUC_teste']>=0.85 else 'NÃO'}")
print(f"   Economia de custo 10:1 vs política atual (th ótimo do treino): {v['Economia_th_ÓTIMO_vs_política']:.2f}%")